#Stage 1 — Description
Label definition and leakage-safe feature split.
There is no ground-truth "needs refresh" column, so a proxy label is required. The dataset provides two non-overlapping windows — prev_30d (older) and last_30d (more recent) — which makes it possible to define the label from an outcome period and restrict features to what was known before that period, rather than mixing the two.
Verification before building anything: trend_pct correlates 1.0000 with the manual (impressions_last_30d − impressions_prev_30d) calculation, confirming it is a direct transformation of the outcome window, not an independent signal. Separately, ctr correlates 0.9999 with clicks_90d / impressions_90d, and the 90-day aggregate columns mathematically contain last_30d inside them. So ctr, avg_position, and every *_90d column are excluded from features — they leak the outcome window.
The label used: needs_refresh = 1 if trend_direction == "down".
The split used: client-holdout (not random rows), because the 32 clients have very uneven page counts (3 to 7,008 pages), and a random split would let one client's pattern leak across train and test.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("content_refresh_anonymized.csv")

# Label from the OUTCOME window (last_30d vs prev_30d)
df["needs_refresh"] = (df["trend_direction"] == "down").astype(int)

# Feature set restricted to the PAST window only — excludes anything
# that overlaps with or defines the outcome window above
safe_features = [
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "age_tier_order", "days_since_last_update",
    "word_count", "char_count",
    "content_type", "main_intent",
    "search_volume", "competition", "cpc",
    "client_id",  # needed for the split, not a model feature
]

df["ctr_prev_30d"] = df["clicks_prev_30d"] / df["impressions_prev_30d"].replace(0, np.nan)
safe_features.append("ctr_prev_30d")

rng = np.random.RandomState(42)
clients = df["client_id"].unique()
test_clients = rng.choice(clients, size=max(1, int(len(clients) * 0.25)), replace=False)

train_df = df[~df["client_id"].isin(test_clients)].copy()   # .copy() avoids pandas SettingWithCopyWarning later
test_df  = df[df["client_id"].isin(test_clients)].copy()

print(f"Train: {train_df.shape[0]} rows, {train_df['client_id'].nunique()} clients")
print(f"Test:  {test_df.shape[0]} rows, {test_df['client_id'].nunique()} clients")
print(f"\nLabel balance — train: {train_df['needs_refresh'].mean():.2%}, test: {test_df['needs_refresh'].mean():.2%}")

Train: 26494 rows, 24 clients
Test:  3506 rows, 8 clients

Label balance — train: 54.45%, test: 52.37%


Stage 2 — Description
Missing-data handling.
Several features have real missingness: word_count/char_count (~26%), search_volume/competition/cpc (~8%), main_intent (~8%). Two rules applied: numeric gaps are filled with the train set's median only (never test, to avoid leaking test statistics into the fill values), plus a _was_missing flag per column, since absence of data can itself be informative. Categorical gaps become an explicit "unknown" category rather than dropped rows — dropping would bias the dataset toward whichever content happens to be well-tracked.

In [ ]:
for d in (train_df, test_df):
    d["ctr_prev_30d"] = d["clicks_prev_30d"] / d["impressions_prev_30d"].replace(0, np.nan)

numeric_cols = ["word_count", "char_count", "search_volume", "competition", "cpc", "ctr_prev_30d"]
categorical_cols = ["content_type", "main_intent"]

for col in numeric_cols:
    median_value = train_df[col].median()  # TRAIN ONLY

    train_df[f"{col}_was_missing"] = train_df[col].isna().astype(int)
    test_df[f"{col}_was_missing"] = test_df[col].isna().astype(int)

    train_df[col] = train_df[col].fillna(median_value)
    test_df[col] = test_df[col].fillna(median_value)  # same train median applied to test

for col in categorical_cols:
    train_df[col] = train_df[col].fillna("unknown")
    test_df[col] = test_df[col].fillna("unknown")

final_features = (
    numeric_cols
    + [f"{c}_was_missing" for c in numeric_cols]
    + categorical_cols
    + ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
       "content_age_days", "age_tier_order", "days_since_last_update"]
)

print("Remaining missing values, train:", train_df[final_features].isna().sum().sum())
print("Remaining missing values, test: ", test_df[final_features].isna().sum().sum())

Remaining missing values, train: 0
Remaining missing values, test:  0


Stage 3 — Description
Hand-rule baseline + Precision@K.
Before any model, a transparent, non-fitted baseline establishes the number a model has to justify itself against. The rule combines three intuitive signals a content strategist could compute without ML: older content, weaker prior CTR, and higher keyword competition all point toward refresh priority. Normalization uses train-set min/max only, to keep test statistics from leaking in. Precision@50 is the eval metric: of the top 50 ranked pages, what fraction were truly labeled needs_refresh. Context worth keeping in mind when reading the result: the test set's base rate is 52.37% needs_refresh, so a meaningless random ranking would already score near that by chance — the baseline needs to clear that floor by a real margin to be worth beating.

In [ ]:
def baseline_score(d, source=train_df):
    def normalize(col):
        lo, hi = source[col].min(), source[col].max()
        return (d[col] - lo) / (hi - lo + 1e-9)

    age_score = normalize("content_age_days")
    ctr_score = 1 - normalize("ctr_prev_30d")
    comp_score = normalize("competition")
    return (age_score + ctr_score + comp_score) / 3

train_df["baseline_score"] = baseline_score(train_df)
test_df["baseline_score"] = baseline_score(test_df)

def precision_at_k(d, score_col, label_col="needs_refresh", k=50):
    top_k = d.sort_values(score_col, ascending=False).head(k)
    return top_k[label_col].mean()

baseline_p50 = precision_at_k(test_df, "baseline_score", k=50)
print(f"Baseline Precision@50: {baseline_p50:.2%}")

Baseline Precision@50: 44.00%


Stage 3b — Description
Diagnosing the baseline result before trusting it.
Precision@50 alone is noisy at this sample size and doesn't tell us whether the rule has any real ranking signal. Two checks: (1) Precision@K at several K values, to see if the below-chance result holds up or was a small-sample fluke; (2) ROC-AUC between baseline_score and needs_refresh across the entire test set — AUC = 0.5 means no better than random, > 0.5 means the rule has real (if weak) separating power, < 0.5 means it's pointing the wrong direction entirely.

In [ ]:
from sklearn.metrics import roc_auc_score

# Precision@K across several K values, to see if 44% at K=50 was noise or a pattern
for k in [25, 50, 100, 200]:
    p = precision_at_k(test_df, "baseline_score", k=k)
    print(f"Baseline Precision@{k}: {p:.2%}")

# AUC uses the WHOLE test set, not just the top K - much more stable read
# on whether baseline_score separates the two classes at all
auc = roc_auc_score(test_df["needs_refresh"], test_df["baseline_score"])
print(f"\nBaseline ROC-AUC: {auc:.4f}  (0.5 = no signal, >0.5 = real signal, <0.5 = inverted)")

Baseline Precision@25: 44.00%
Baseline Precision@50: 44.00%
Baseline Precision@100: 55.00%
Baseline Precision@200: 63.50%

Baseline ROC-AUC: 0.4663  (0.5 = no signal, >0.5 = real signal, <0.5 = inverted)


Stage 3c — Description
Decomposing the baseline to find which ingredient is actually broken.
The combined score blends three signals equally, which can hide the fact that one is doing real work while another is actively wrong. Check each raw signal's individual relationship to the label via AUC, and check for tie-clustering at the top of the ranking (a sign that "top 25" isn't a meaningful ranking at all, just an artifact of duplicate values).

In [ ]:
from sklearn.metrics import roc_auc_score

# --- Check each raw ingredient separately against the label ---
# If one of these is near/below 0.5, that ingredient is dead weight or actively
# wrong in the blended score - not just "weak," but worth dropping or flipping.
for col, direction in [("content_age_days", "higher = more refresh?"),
                       ("ctr_prev_30d", "LOWER = more refresh? (we flipped this one)"),
                       ("competition", "higher = more refresh?")]:
    auc = roc_auc_score(test_df["needs_refresh"], test_df[col])
    print(f"{col:20s} AUC: {auc:.4f}   ({direction})")

# --- Check for tie-clustering at the top of the ranking ---
# If many rows share the exact same top score, "top 25" is partly arbitrary,
# not a real ranking - this would explain a noisy/bad Precision@25 on its own.
top25 = test_df.sort_values("baseline_score", ascending=False).head(25)
print(f"\nUnique baseline_score values in top 25: {top25['baseline_score'].nunique()} (out of 25)")
print(f"Unique baseline_score values overall:    {test_df['baseline_score'].nunique()} (out of {len(test_df)})")

content_age_days     AUC: 0.4552   (higher = more refresh?)
ctr_prev_30d         AUC: 0.5172   (LOWER = more refresh? (we flipped this one))
competition          AUC: 0.5045   (higher = more refresh?)

Unique baseline_score values in top 25: 17 (out of 25)
Unique baseline_score values overall:    1420 (out of 3506)


Stage 3d — Description
Correcting the baseline without touching the test set to do it.
Re-run the direction check on train_df only. Any signal whose train AUC is essentially 0.5 (within ~0.02) gets dropped entirely rather than kept as noise. The surviving signals get oriented based on train, then the corrected rule is evaluated on test_df a single time — this is the legitimate version of the "fix," not the version that peeked at test answers first.

In [ ]:
from sklearn.metrics import roc_auc_score
import pandas as pd

candidate_cols = ["content_age_days", "ctr_prev_30d", "competition"]
signal_info = {}

# Direction + strength decided from TRAIN only
for col in candidate_cols:
    auc = roc_auc_score(train_df["needs_refresh"], train_df[col])
    signal_info[col] = {"auc": auc, "flip": auc < 0.5, "strength": abs(auc - 0.5)}
    print(f"{col:20s} train AUC: {auc:.4f}  strength: {signal_info[col]['strength']:.4f}  "
          f"{'(flip)' if signal_info[col]['flip'] else '(keep)'}")

# Drop anything too close to 0.5 - it's noise, not signal
keep_cols = [c for c, info in signal_info.items() if info["strength"] > 0.02]
print(f"\nKeeping: {keep_cols}")

def normalize(d, col, source=train_df):
    lo, hi = source[col].min(), source[col].max()
    norm = (d[col] - lo) / (hi - lo + 1e-9)
    return (1 - norm) if signal_info[col]["flip"] else norm

def corrected_baseline_score(d):
    if not keep_cols:
        return pd.Series(0.5, index=d.index)  # honestly: no usable signal at all
    return sum(normalize(d, c) for c in keep_cols) / len(keep_cols)

train_df["baseline_score_v2"] = corrected_baseline_score(train_df)
test_df["baseline_score_v2"] = corrected_baseline_score(test_df)

# Single, final check on test - not re-diagnosed, just evaluated
for k in [25, 50, 100, 200]:
    p = precision_at_k(test_df, "baseline_score_v2", k=k)
    print(f"Corrected baseline Precision@{k}: {p:.2%}")

auc_v2 = roc_auc_score(test_df["needs_refresh"], test_df["baseline_score_v2"])
print(f"\nCorrected baseline ROC-AUC: {auc_v2:.4f}")

content_age_days     train AUC: 0.3982  strength: 0.1018  (flip)
ctr_prev_30d         train AUC: 0.5193  strength: 0.0193  (keep)
competition          train AUC: 0.4975  strength: 0.0025  (flip)

Keeping: ['content_age_days']
Corrected baseline Precision@25: 60.00%
Corrected baseline Precision@50: 54.00%
Corrected baseline Precision@100: 56.00%
Corrected baseline Precision@200: 55.50%

Corrected baseline ROC-AUC: 0.5448


Stage 4 — Description
First learned model: Decision Tree.
A Decision Tree can combine all the safe features at once — not just one signal at a time like the hand-rule could — which is exactly the "ML beats a fixed rule" case being made. Categorical columns (content_type, main_intent) need one-hot encoding since trees require numeric input; encoding is fit on train categories only, and test is aligned to the same columns afterward, so an unseen category in test can't silently create a mismatched shape or leak train-only category info. Tree depth is capped deliberately — a very deep tree would overfit on ~26k rows and 24 clients, and it would stop being interpretable, which matters given the preference for simple/explainable models over accuracy-maximizing ones.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd

model_features = final_features  # from Stage 2: numeric + missing-flags + past-window raw cols
categorical_cols = ["content_type", "main_intent"]
numeric_and_flag_cols = [c for c in model_features if c not in categorical_cols]

# One-hot encode categoricals - FIT on train, then align test to the same columns
train_dummies = pd.get_dummies(train_df[categorical_cols], prefix=categorical_cols)
test_dummies = pd.get_dummies(test_df[categorical_cols], prefix=categorical_cols)
test_dummies = test_dummies.reindex(columns=train_dummies.columns, fill_value=0)
# reindex matters: if test has a content_type train never saw, this keeps shapes aligned
# instead of crashing or silently misaligning columns

X_train = pd.concat([train_df[numeric_and_flag_cols].reset_index(drop=True), train_dummies.reset_index(drop=True)], axis=1)
X_test  = pd.concat([test_df[numeric_and_flag_cols].reset_index(drop=True), test_dummies.reset_index(drop=True)], axis=1)
y_train = train_df["needs_refresh"].reset_index(drop=True)
y_test  = test_df["needs_refresh"].reset_index(drop=True)

tree = DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced")
tree.fit(X_train, y_train)

test_df = test_df.reset_index(drop=True)
test_df["tree_score"] = tree.predict_proba(X_test)[:, 1]

for k in [25, 50, 100, 200]:
    p = precision_at_k(test_df, "tree_score", k=k)
    print(f"Tree Precision@{k}: {p:.2%}")

auc_tree = roc_auc_score(y_test, test_df["tree_score"])
print(f"\nTree ROC-AUC: {auc_tree:.4f}   (baseline was 0.5448)")

# Interpretability check - which features actually drove the tree's decisions
importances = pd.Series(tree.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("\nTop 10 feature importances:")
print(importances.head(10))

Tree Precision@25: 68.00%
Tree Precision@50: 58.00%
Tree Precision@100: 60.00%
Tree Precision@200: 62.50%

Tree ROC-AUC: 0.7351   (baseline was 0.5448)

Top 10 feature importances:
ctr_prev_30d_was_missing        0.600740
content_age_days                0.198149
clicks_prev_30d                 0.054759
impressions_prev_30d            0.044503
days_since_last_update          0.039163
word_count                      0.034115
content_type_keyword article    0.015020
char_count                      0.011811
search_volume                   0.001740
competition                     0.000000
dtype: float64


Stage 4b — Description
Checking whether the top feature is a real signal or a scope artifact.
Crosstab the missingness flag against the original multi-class trend_direction (not the collapsed binary label) to see if missing-prior-impressions rows are dominated by "new". If so, the honest fix isn't to drop the feature — it's to reconsider whether brand-new pages belong in this population at all. "Should this page be refreshed" isn't a meaningful question for content with no performance history yet; the right move is likely excluding trend_direction == "new" from the modeling population entirely, not labeling it 0 by default.

In [ ]:
print("trend_direction breakdown WHERE ctr_prev_30d_was_missing == 1 (train):")
print(train_df.loc[train_df["ctr_prev_30d_was_missing"] == 1, "trend_direction"].value_counts())

print("\ntrend_direction breakdown WHERE ctr_prev_30d_was_missing == 0 (train):")
print(train_df.loc[train_df["ctr_prev_30d_was_missing"] == 0, "trend_direction"].value_counts())

print("\nneeds_refresh rate | missing flag = 1:", train_df.loc[train_df["ctr_prev_30d_was_missing"] == 1, "needs_refresh"].mean())
print("needs_refresh rate | missing flag = 0:", train_df.loc[train_df["ctr_prev_30d_was_missing"] == 0, "needs_refresh"].mean())
print("\nShare of train rows with missing flag = 1:", train_df["ctr_prev_30d_was_missing"].mean())

trend_direction breakdown WHERE ctr_prev_30d_was_missing == 1 (train):
trend_direction
new     1894
flat     806
Name: count, dtype: int64

trend_direction breakdown WHERE ctr_prev_30d_was_missing == 0 (train):
trend_direction
down      14426
stable     5517
up         3851
Name: count, dtype: int64

needs_refresh rate | missing flag = 1: 0.0
needs_refresh rate | missing flag = 0: 0.606287299319156

Share of train rows with missing flag = 1: 0.10190986638484185


Stage 5 — Description
Correcting the modeling population. Excludes trend_direction values of "new" and "flat" before anything else happens, since these represent out-of-scope questions (no history / no visibility) rather than genuine refresh-decisions. Label, split, and missing-data handling are all rebuilt from scratch on this corrected population — reusing stale computations from the flawed population would carry the same artifact forward.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("content_refresh_anonymized.csv")

# --- Scope correction: exclude pages with no track record or no visibility ---
# "new" = no performance history yet; "flat" = zero impressions in both windows
# (invisible to search - a discoverability problem, not a refresh problem).
# Neither is a valid case for "does this page need a content refresh?"
before = len(df)
df = df[~df["trend_direction"].isin(["new", "flat"])].copy()
print(f"Dropped {before - len(df)} rows ({(before - len(df))/before:.1%}) - new/flat pages")
print(f"Remaining population: {len(df)} rows")

# --- Label, rebuilt on the corrected population ---
df["needs_refresh"] = (df["trend_direction"] == "down").astype(int)
print(f"New label balance: {df['needs_refresh'].mean():.2%}")  # expect ~61%, not 54% - a smaller, cleaner population

# --- ctr_prev_30d, rebuilt ---
df["ctr_prev_30d"] = df["clicks_prev_30d"] / df["impressions_prev_30d"].replace(0, np.nan)

# --- Client-holdout split, same logic as Stage 1 ---
rng = np.random.RandomState(42)
clients = df["client_id"].unique()
test_clients = rng.choice(clients, size=max(1, int(len(clients) * 0.25)), replace=False)

train_df = df[~df["client_id"].isin(test_clients)].copy()
test_df  = df[df["client_id"].isin(test_clients)].copy()

print(f"\nTrain: {train_df.shape[0]} rows, {train_df['client_id'].nunique()} clients")
print(f"Test:  {test_df.shape[0]} rows, {test_df['client_id'].nunique()} clients")
print(f"Label balance — train: {train_df['needs_refresh'].mean():.2%}, test: {test_df['needs_refresh'].mean():.2%}")

# --- Missing data, rebuilt on the corrected population ---
# Check first: does ctr_prev_30d still have missingness now that new/flat are gone?
print(f"\nRemaining missing ctr_prev_30d, train: {train_df['ctr_prev_30d'].isna().sum()} / {len(train_df)}")

numeric_cols = ["word_count", "char_count", "search_volume", "competition", "cpc", "ctr_prev_30d"]
categorical_cols = ["content_type", "main_intent"]

for col in numeric_cols:
    median_value = train_df[col].median()
    train_df[f"{col}_was_missing"] = train_df[col].isna().astype(int)
    test_df[f"{col}_was_missing"] = test_df[col].isna().astype(int)
    train_df[col] = train_df[col].fillna(median_value)
    test_df[col] = test_df[col].fillna(median_value)

for col in categorical_cols:
    train_df[col] = train_df[col].fillna("unknown")
    test_df[col] = test_df[col].fillna("unknown")

final_features = (
    numeric_cols
    + [f"{c}_was_missing" for c in numeric_cols]
    + categorical_cols
    + ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
       "content_age_days", "age_tier_order", "days_since_last_update"]
)

print("Remaining missing values, train:", train_df[final_features].isna().sum().sum())
print("Remaining missing values, test: ", test_df[final_features].isna().sum().sum())

Dropped 3388 rows (11.3%) - new/flat pages
Remaining population: 26612 rows
New label balance: 61.11%

Train: 23778 rows, 24 clients
Test:  2834 rows, 7 clients
Label balance — train: 59.97%, test: 70.68%

Remaining missing ctr_prev_30d, train: 0 / 23778
Remaining missing values, train: 0
Remaining missing values, test:  0


Stage 5b — Description
Checking split stability across seeds, selecting for balance, not performance. Try several random seeds for the client-holdout split and compare train/test label-balance gaps and client counts. Pick the seed with the smallest, most reasonable gap — decided purely from label statistics, before any model is trained on it, so this isn't performance-motivated seed shopping.

In [ ]:
import numpy as np

clients = df["client_id"].unique()
print(f"Total unique clients in corrected population: {len(clients)}")

for seed in [0, 1, 7, 42, 99]:
    rng = np.random.RandomState(seed)
    test_clients = rng.choice(clients, size=max(1, int(len(clients) * 0.25)), replace=False)

    tr = df[~df["client_id"].isin(test_clients)]
    te = df[df["client_id"].isin(test_clients)]

    gap = abs(tr["needs_refresh"].mean() - te["needs_refresh"].mean())
    print(f"seed={seed:3d} | train n={len(tr):5d} clients={tr['client_id'].nunique():2d} "
          f"bal={tr['needs_refresh'].mean():.2%} | test n={len(te):5d} clients={te['client_id'].nunique():2d} "
          f"bal={te['needs_refresh'].mean():.2%} | gap={gap:.2%}")

Total unique clients in corrected population: 31
seed=  0 | train n=24421 clients=24 bal=59.80% | test n= 2191 clients= 7 bal=75.72% | gap=15.92%
seed=  1 | train n=18546 clients=24 bal=66.29% | test n= 8066 clients= 7 bal=49.19% | gap=17.10%
seed=  7 | train n=21354 clients=24 bal=59.50% | test n= 5258 clients= 7 bal=67.65% | gap=8.15%
seed= 42 | train n=23778 clients=24 bal=59.97% | test n= 2834 clients= 7 bal=70.68% | gap=10.71%
seed= 99 | train n=24354 clients=24 bal=60.83% | test n= 2258 clients= 7 bal=64.08% | gap=3.25%


Stage 6 — Description
Lock in seed=99 and rebuild the corrected pipeline on it. Same logic as Stage 5 (scope correction, label, missing-data handling) — only the split seed changes, from 42 to 99. This becomes the split we iterate on and eventually report.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("content_refresh_anonymized.csv")

# Scope correction (same as Stage 5)
df = df[~df["trend_direction"].isin(["new", "flat"])].copy()
df["needs_refresh"] = (df["trend_direction"] == "down").astype(int)
df["ctr_prev_30d"] = df["clicks_prev_30d"] / df["impressions_prev_30d"].replace(0, np.nan)

# Split — LOCKED to seed=99, chosen for balance in Stage 5b, before any model existed
rng = np.random.RandomState(99)
clients = df["client_id"].unique()
test_clients = rng.choice(clients, size=max(1, int(len(clients) * 0.25)), replace=False)

train_df = df[~df["client_id"].isin(test_clients)].copy()
test_df  = df[df["client_id"].isin(test_clients)].copy()

print(f"Train: {train_df.shape[0]} rows, {train_df['client_id'].nunique()} clients, bal={train_df['needs_refresh'].mean():.2%}")
print(f"Test:  {test_df.shape[0]} rows, {test_df['client_id'].nunique()} clients, bal={test_df['needs_refresh'].mean():.2%}")

# Missing data (same logic as Stage 5)
numeric_cols = ["word_count", "char_count", "search_volume", "competition", "cpc", "ctr_prev_30d"]
categorical_cols = ["content_type", "main_intent"]

for col in numeric_cols:
    median_value = train_df[col].median()
    train_df[f"{col}_was_missing"] = train_df[col].isna().astype(int)
    test_df[f"{col}_was_missing"] = test_df[col].isna().astype(int)
    train_df[col] = train_df[col].fillna(median_value)
    test_df[col] = test_df[col].fillna(median_value)

for col in categorical_cols:
    train_df[col] = train_df[col].fillna("unknown")
    test_df[col] = test_df[col].fillna("unknown")

final_features = (
    numeric_cols
    + [f"{c}_was_missing" for c in numeric_cols]
    + categorical_cols
    + ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
       "content_age_days", "age_tier_order", "days_since_last_update"]
)

print("Remaining missing values, train:", train_df[final_features].isna().sum().sum())
print("Remaining missing values, test: ", test_df[final_features].isna().sum().sum())

Train: 24354 rows, 24 clients, bal=60.83%
Test:  2258 rows, 7 clients, bal=64.08%
Remaining missing values, train: 0
Remaining missing values, test:  0


Stage 7 — Description
Re-diagnosing baseline signals on the corrected population. The scope change (removing new/flat pages) may have changed which raw signals actually relate to needs_refresh — the old diagnosis (Stage 3d) was run on a population that doesn't exist anymore. Rather than assume content_age_days is still the only real signal, sweep a broader set of candidate columns and check each one's AUC against train only. This also widens the net a bit versus the original 3-ingredient guess, since a wider sweep is more honest than re-testing the same three we happened to pick first.

In [ ]:
from sklearn.metrics import roc_auc_score

# Wider candidate sweep than Stage 3d - the population changed, so don't assume
# the same 3 features are still the only (or the right) candidates
candidate_cols = [
    "content_age_days", "ctr_prev_30d", "competition",
    "word_count", "char_count", "search_volume", "cpc",
    "days_since_last_update", "impressions_prev_30d", "clicks_prev_30d",
]

signal_info = {}
for col in candidate_cols:
    auc = roc_auc_score(train_df["needs_refresh"], train_df[col])
    signal_info[col] = {"auc": auc, "flip": auc < 0.5, "strength": abs(auc - 0.5)}

results = pd.DataFrame(signal_info).T.sort_values("strength", ascending=False)
print(results)

keep_cols = results[results["strength"] > 0.02].index.tolist()
print(f"\nKeeping (strength > 0.02): {keep_cols}")

                             auc  flip  strength
content_age_days        0.388002  True  0.111998
search_volume           0.443505  True  0.056495
clicks_prev_30d         0.454881  True  0.045119
ctr_prev_30d            0.455198  True  0.044802
days_since_last_update  0.476946  True  0.023054
competition             0.487428  True  0.012572
cpc                     0.489131  True  0.010869
char_count              0.490689  True  0.009311
word_count              0.493909  True  0.006091
impressions_prev_30d    0.494108  True  0.005892

Keeping (strength > 0.02): ['content_age_days', 'search_volume', 'clicks_prev_30d', 'ctr_prev_30d', 'days_since_last_update']


Stage 8 — Description
Rebuild the baseline (v3) using the five validated signals, on the corrected population. Same equal-weighted blend methodology as Stage 3d, applied to the newly-confirmed signal set. Normalization stats come from train only. This becomes the number the rebuilt Decision Tree has to beat on the corrected population and locked split — not the old, artifact-inflated 0.7351.

In [ ]:
from sklearn.metrics import roc_auc_score

keep_cols = ["content_age_days", "search_volume", "clicks_prev_30d", "ctr_prev_30d", "days_since_last_update"]

# Confirm direction on THIS train set (don't assume the Stage 7 flip=True holds forever)
flip_map = {}
for col in keep_cols:
    auc = roc_auc_score(train_df["needs_refresh"], train_df[col])
    flip_map[col] = auc < 0.5

def normalize(d, col, source=train_df):
    lo, hi = source[col].min(), source[col].max()
    norm = (d[col] - lo) / (hi - lo + 1e-9)
    return (1 - norm) if flip_map[col] else norm

def baseline_score_v3(d):
    return sum(normalize(d, c) for c in keep_cols) / len(keep_cols)

train_df["baseline_score_v3"] = baseline_score_v3(train_df)
test_df["baseline_score_v3"] = baseline_score_v3(test_df)

for k in [25, 50, 100, 200]:
    p = precision_at_k(test_df, "baseline_score_v3", k=k)
    print(f"Baseline v3 Precision@{k}: {p:.2%}")

auc_v3 = roc_auc_score(test_df["needs_refresh"], test_df["baseline_score_v3"])
print(f"\nBaseline v3 ROC-AUC: {auc_v3:.4f}   (test base rate: {test_df['needs_refresh'].mean():.2%})")

Baseline v3 Precision@25: 100.00%
Baseline v3 Precision@50: 92.00%
Baseline v3 Precision@100: 83.00%
Baseline v3 Precision@200: 76.50%

Baseline v3 ROC-AUC: 0.6024   (test base rate: 64.08%)


Stage 8b — Description
Sanity-checking baseline v3 before accepting it. Two checks: (1) how many unique score values exist in the top 25/50 — low uniqueness would mean the "100%" is partly an artifact of ties, not a sharp ranking; (2) how much each of the five normalized components actually varies and correlates with the final blended score and the label — this reveals whether one feature is quietly dominating the equal-weighted sum (min-max normalization bounds each feature to [0,1], but a heavy-tailed feature can still end up contributing far more to the ranking than the others, even at "equal weight").

--- Tie-clustering check ---

In [ ]:
top25 = test_df.sort_values("baseline_score_v3", ascending=False).head(25)
top50 = test_df.sort_values("baseline_score_v3", ascending=False).head(50)
print(f"Unique scores in top 25: {top25['baseline_score_v3'].nunique()} / 25")
print(f"Unique scores in top 50: {top50['baseline_score_v3'].nunique()} / 50")

# --- Which component is actually driving the ranking? ---
print("\nNormalized component behavior on test_df:")
for col in keep_cols:
    norm_col = normalize(test_df, col)  # reuse the function from Stage 8
    std = norm_col.std()
    corr_with_blend = norm_col.corr(test_df["baseline_score_v3"])
    corr_with_label = norm_col.corr(test_df["needs_refresh"].astype(float))
    print(f"{col:25s} std={std:.4f}  corr_w_blend={corr_with_blend:.3f}  corr_w_label={corr_with_label:.3f}")

Unique scores in top 25: 5 / 25
Unique scores in top 50: 13 / 50

Normalized component behavior on test_df:
content_age_days          std=0.3063  corr_w_blend=0.917  corr_w_label=0.164
search_volume             std=0.0127  corr_w_blend=0.064  corr_w_label=0.002
clicks_prev_30d           std=0.0033  corr_w_blend=-0.050  corr_w_label=0.035
ctr_prev_30d              std=0.1104  corr_w_blend=0.375  corr_w_label=0.085
days_since_last_update    std=0.0773  corr_w_blend=0.280  corr_w_label=0.055


Stage 8c — Description
Fixing the baseline with rank-based (percentile) normalization instead of min-max. Rank normalization converts each value to "what fraction of the train distribution is at or below this," which is naturally bounded to [0,1] and immune to outlier compression — a page doesn't get flattened into the same bucket as thousands of others just because one competitor had a freak traffic spike. Ranks are computed against the train distribution only, applied to both train and test, so test values are scored relative to what train saw, not re-ranked against themselves.

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

keep_cols = ["content_age_days", "search_volume", "clicks_prev_30d", "ctr_prev_30d", "days_since_last_update"]

def rank_normalize(d, col, source=train_df):
    # Percentile rank against the TRAIN distribution - robust to outliers,
    # unlike min-max which lets one extreme value crush everything else near 0
    sorted_source = np.sort(source[col].values)
    ranks = np.searchsorted(sorted_source, d[col].values, side="right")
    return pd.Series(ranks / len(sorted_source), index=d.index)  # <-- wrap as Series

flip_map = {col: roc_auc_score(train_df["needs_refresh"], train_df[col]) < 0.5 for col in keep_cols}

def baseline_score_v4(d):
    total = pd.Series(0.0, index=d.index)
    for col in keep_cols:
        r = rank_normalize(d, col)
        total += (1 - r) if flip_map[col] else r
    return total / len(keep_cols)

train_df["baseline_score_v4"] = baseline_score_v4(train_df)
test_df["baseline_score_v4"] = baseline_score_v4(test_df)

# Re-run the SAME sanity checks as Stage 8b, to confirm the fix actually worked
top25 = test_df.sort_values("baseline_score_v4", ascending=False).head(25)
top50 = test_df.sort_values("baseline_score_v4", ascending=False).head(50)
print(f"Unique scores in top 25: {top25['baseline_score_v4'].nunique()} / 25")
print(f"Unique scores in top 50: {top50['baseline_score_v4'].nunique()} / 50")

print("\nComponent behavior on test_df (rank-normalized):")
for col in keep_cols:
    r = rank_normalize(test_df, col)
    comp = (1 - r) if flip_map[col] else r
    print(f"{col:25s} std={comp.std():.4f}  corr_w_blend={comp.corr(test_df['baseline_score_v4']):.3f}")

for k in [25, 50, 100, 200]:
    print(f"Baseline v4 Precision@{k}: {precision_at_k(test_df, 'baseline_score_v4', k=k):.2%}")

auc_v4 = roc_auc_score(test_df["needs_refresh"], test_df["baseline_score_v4"])
print(f"\nBaseline v4 ROC-AUC: {auc_v4:.4f}")

Unique scores in top 25: 13 / 25
Unique scores in top 50: 18 / 50

Component behavior on test_df (rank-normalized):
content_age_days          std=0.3064  corr_w_blend=0.700
search_volume             std=0.1787  corr_w_blend=0.388
clicks_prev_30d           std=0.0890  corr_w_blend=0.369
ctr_prev_30d              std=0.1587  corr_w_blend=0.433
days_since_last_update    std=0.1498  corr_w_blend=0.444
Baseline v4 Precision@25: 84.00%
Baseline v4 Precision@50: 92.00%
Baseline v4 Precision@100: 85.00%
Baseline v4 Precision@200: 79.00%

Baseline v4 ROC-AUC: 0.6072


Stage 9 — Description
Rebuild the Decision Tree on the corrected population + locked split (seed 99), and check feature importances immediately this time — not as a follow-up after something looked suspicious, but as a standard part of running any model from now on, since that's exactly the step that caught the "new"/"flat" leak two stages ago.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd

categorical_cols = ["content_type", "main_intent"]
numeric_and_flag_cols = [c for c in final_features if c not in categorical_cols]

train_dummies = pd.get_dummies(train_df[categorical_cols], prefix=categorical_cols)
test_dummies = pd.get_dummies(test_df[categorical_cols], prefix=categorical_cols)
test_dummies = test_dummies.reindex(columns=train_dummies.columns, fill_value=0)

X_train = pd.concat([train_df[numeric_and_flag_cols].reset_index(drop=True), train_dummies.reset_index(drop=True)], axis=1)
X_test  = pd.concat([test_df[numeric_and_flag_cols].reset_index(drop=True), test_dummies.reset_index(drop=True)], axis=1)
y_train = train_df["needs_refresh"].reset_index(drop=True)
y_test  = test_df["needs_refresh"].reset_index(drop=True)

tree = DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced")
tree.fit(X_train, y_train)

test_df = test_df.reset_index(drop=True)
test_df["tree_score"] = tree.predict_proba(X_test)[:, 1]

for k in [25, 50, 100, 200]:
    print(f"Tree Precision@{k}: {precision_at_k(test_df, 'tree_score', k=k):.2%}")

auc_tree = roc_auc_score(y_test, test_df["tree_score"])
print(f"\nTree ROC-AUC: {auc_tree:.4f}   (baseline v4 was 0.6072)")

importances = pd.Series(tree.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("\nTop 10 feature importances:")
print(importances.head(10))

# Built-in artifact check, done proactively this time
top_feature = importances.index[0]
print(f"\nTop feature: '{top_feature}' — {importances.iloc[0] / importances.sum():.1%} of total importance")

Tree Precision@25: 96.00%
Tree Precision@50: 72.00%
Tree Precision@100: 76.00%
Tree Precision@200: 81.00%

Tree ROC-AUC: 0.6788   (baseline v4 was 0.6072)

Top 10 feature importances:
content_age_days             0.496020
clicks_prev_30d              0.141354
word_count                   0.089398
days_since_last_update       0.088407
impressions_prev_30d         0.066545
ctr_prev_30d                 0.060305
char_count_was_missing       0.023239
char_count                   0.020034
sessions_prev_30d            0.007002
main_intent_transactional    0.003694
dtype: float64

Top feature: 'content_age_days' — 49.6% of total importance


Stage 9b — Description
Overfitting check + cross-client stability via GroupKFold. First: compare train vs. test AUC on the current locked split — a large gap would mean the tree memorized rather than generalized. Second: run 5-fold GroupKFold (splitting by client_id, so no client ever appears in both train and test within a fold) to see whether ~0.68 AUC and content_age_days's dominance hold up across different client groupings, not just this one split.

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np

# --- 1. Overfitting check on the current locked split ---
train_pred = tree.predict_proba(X_train)[:, 1]
train_auc = roc_auc_score(y_train, train_pred)
print(f"Train AUC: {train_auc:.4f}   Test AUC: {auc_tree:.4f}   Gap: {train_auc - auc_tree:.4f}")
# A gap under ~0.05-0.08 is generally healthy for a shallow tree; much bigger
# than that means it's fitting to noise/client quirks, not real patterns.

# --- 2. Cross-client stability via GroupKFold ---
# Reusable prep function so each fold's imputation/encoding is fit on THAT
# fold's train data only - same leakage discipline as before, just repeated 5x
def prepare_features(train_raw, test_raw):
    train_raw, test_raw = train_raw.copy(), test_raw.copy()
    numeric_cols = ["word_count", "char_count", "search_volume", "competition", "cpc", "ctr_prev_30d"]
    categorical_cols = ["content_type", "main_intent"]
    for col in numeric_cols:
        med = train_raw[col].median()
        train_raw[f"{col}_was_missing"] = train_raw[col].isna().astype(int)
        test_raw[f"{col}_was_missing"] = test_raw[col].isna().astype(int)
        train_raw[col] = train_raw[col].fillna(med)
        test_raw[col] = test_raw[col].fillna(med)
    for col in categorical_cols:
        train_raw[col] = train_raw[col].fillna("unknown")
        test_raw[col] = test_raw[col].fillna("unknown")
    feats = (numeric_cols + [f"{c}_was_missing" for c in numeric_cols] + categorical_cols
             + ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
                "content_age_days", "age_tier_order", "days_since_last_update"])
    num_flag_cols = [c for c in feats if c not in categorical_cols]
    tr_dum = pd.get_dummies(train_raw[categorical_cols], prefix=categorical_cols)
    te_dum = pd.get_dummies(test_raw[categorical_cols], prefix=categorical_cols).reindex(columns=tr_dum.columns, fill_value=0)
    Xtr = pd.concat([train_raw[num_flag_cols].reset_index(drop=True), tr_dum.reset_index(drop=True)], axis=1)
    Xte = pd.concat([test_raw[num_flag_cols].reset_index(drop=True), te_dum.reset_index(drop=True)], axis=1)
    return Xtr, train_raw["needs_refresh"].reset_index(drop=True), Xte, test_raw["needs_refresh"].reset_index(drop=True)

gkf = GroupKFold(n_splits=5)
fold_aucs = []
for fold, (tr_idx, te_idx) in enumerate(gkf.split(df, groups=df["client_id"])):
    Xtr, ytr, Xte, yte = prepare_features(df.iloc[tr_idx], df.iloc[te_idx])
    t = DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced")
    t.fit(Xtr, ytr)
    fold_auc = roc_auc_score(yte, t.predict_proba(Xte)[:, 1])
    fold_aucs.append(fold_auc)
    print(f"Fold {fold}: test clients={df.iloc[te_idx]['client_id'].nunique()}, AUC={fold_auc:.4f}")

print(f"\nMean fold AUC: {np.mean(fold_aucs):.4f}   Std: {np.std(fold_aucs):.4f}   (single-split test was 0.6788)")

Train AUC: 0.6851   Test AUC: 0.6788   Gap: 0.0063
Fold 0: test clients=1, AUC=0.5746
Fold 1: test clients=7, AUC=0.6170
Fold 2: test clients=8, AUC=0.5306
Fold 3: test clients=7, AUC=0.5730
Fold 4: test clients=8, AUC=0.5806

Mean fold AUC: 0.5752   Std: 0.0275   (single-split test was 0.6788)


Stage 9c — Description
Fixing the comparison: cross-validate the baseline the same way, on the same folds, and inspect what's actually in the outlier fold. Whichever model wins should win on identical footing.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# ----------------------------
# Features used in baseline
# ----------------------------
keep_cols = [
    "content_age_days",
    "search_volume",
    "clicks_prev_30d",
    "ctr_prev_30d",
    "days_since_last_update"
]

# ----------------------------
# Show fold information
# ----------------------------
print("=" * 60)
print("Fold Information")
print("=" * 60)

for fold, (tr_idx, te_idx) in enumerate(gkf.split(df, groups=df["client_id"])):
    te = df.iloc[te_idx]

    print(
        f"Fold {fold}: "
        f"{len(te_idx)} rows | "
        f"{te['client_id'].nunique()} clients | "
        f"Largest client = {te['client_id'].value_counts().head(1).to_dict()}"
    )

print("\n")

# ----------------------------
# Rank normalization function
# ----------------------------
def rank_normalize_arr(values, source_sorted):
    if len(source_sorted) == 0:
        return np.zeros(len(values))

    ranks = np.searchsorted(source_sorted, values, side="right")
    return ranks / len(source_sorted)

# ----------------------------
# Cross-validation
# ----------------------------
baseline_fold_aucs = []

print("=" * 60)
print("Baseline Cross Validation")
print("=" * 60)

for fold, (tr_idx, te_idx) in enumerate(gkf.split(df, groups=df["client_id"])):

    tr = df.iloc[tr_idx].copy()
    te = df.iloc[te_idx].copy()

    # -------------------------------------
    # Fill missing values using TRAIN median
    # -------------------------------------
    for col in keep_cols:
        median = tr[col].median()

        tr[col] = tr[col].fillna(median)
        te[col] = te[col].fillna(median)

    # -------------------------------------
    # Skip fold if target has only one class
    # -------------------------------------
    if tr["needs_refresh"].nunique() < 2:
        print(f"Fold {fold}: skipped (training target has one class)")
        continue

    if te["needs_refresh"].nunique() < 2:
        print(f"Fold {fold}: skipped (test target has one class)")
        continue

    # -------------------------------------
    # Determine which features need flipping
    # -------------------------------------
    flip_map_fold = {}

    for col in keep_cols:
        try:
            auc_feature = roc_auc_score(
                tr["needs_refresh"],
                tr[col]
            )

            flip_map_fold[col] = auc_feature < 0.5

        except ValueError:
            # Feature is constant or invalid
            flip_map_fold[col] = False

    # -------------------------------------
    # Build baseline score
    # -------------------------------------
    score = np.zeros(len(te))

    for col in keep_cols:

        sorted_train = np.sort(tr[col].values)

        r = rank_normalize_arr(
            te[col].values,
            sorted_train
        )

        if flip_map_fold[col]:
            score += (1 - r)
        else:
            score += r

    score /= len(keep_cols)

    # -------------------------------------
    # Evaluate fold
    # -------------------------------------
    auc = roc_auc_score(
        te["needs_refresh"],
        score
    )

    baseline_fold_aucs.append(auc)

    print(f"Fold {fold} baseline AUC = {auc:.4f}")

# ----------------------------
# Final Results
# ----------------------------
print("\n" + "=" * 60)

if len(baseline_fold_aucs) > 0:

    print(
        f"Mean baseline CV AUC : {np.mean(baseline_fold_aucs):.4f}"
    )

    print(
        f"Std baseline CV AUC  : {np.std(baseline_fold_aucs):.4f}"
    )

else:
    print("No valid baseline folds were evaluated.")

# ----------------------------
# Compare with Tree Model
# ----------------------------
if "fold_aucs" in globals() and len(fold_aucs) > 0:

    print(
        f"\nMean tree CV AUC     : {np.mean(fold_aucs):.4f}"
    )

    print(
        f"Std tree CV AUC      : {np.std(fold_aucs):.4f}"
    )

    print("\nFold-by-fold comparison")

    for i, (tree_auc, base_auc) in enumerate(zip(fold_aucs, baseline_fold_aucs)):
        print(
            f"Fold {i}: "
            f"Tree={tree_auc:.4f} | "
            f"Baseline={base_auc:.4f} | "
            f"Tree wins={tree_auc > base_auc}"
        )

Fold Information
Fold 0: 6983 rows | 1 clients | Largest client = {'client_19581e27de': 6983}
Fold 1: 4906 rows | 7 clients | Largest client = {'client_6208ef0f77': 3678}
Fold 2: 4906 rows | 8 clients | Largest client = {'client_4e07408562': 2289}
Fold 3: 4907 rows | 7 clients | Largest client = {'client_3fdba35f04': 2262}
Fold 4: 4910 rows | 8 clients | Largest client = {'client_f369cb89fc': 1618}


Baseline Cross Validation
Fold 0 baseline AUC = 0.5553
Fold 1 baseline AUC = 0.5795
Fold 2 baseline AUC = 0.5981
Fold 3 baseline AUC = 0.5288
Fold 4 baseline AUC = 0.5880

Mean baseline CV AUC : 0.5699
Std baseline CV AUC  : 0.0250

Mean tree CV AUC     : 0.5752
Std tree CV AUC      : 0.0275

Fold-by-fold comparison
Fold 0: Tree=0.5746 | Baseline=0.5553 | Tree wins=True
Fold 1: Tree=0.6170 | Baseline=0.5795 | Tree wins=True
Fold 2: Tree=0.5306 | Baseline=0.5981 | Tree wins=False
Fold 3: Tree=0.5730 | Baseline=0.5288 | Tree wins=True
Fold 4: Tree=0.5806 | Baseline=0.5880 | Tree wins=False


Stage 10 — Description
Random Forest, cross-validated on the same GroupKFold folds as the tree and baseline — same features, same leakage discipline (per-fold train-only imputation/encoding), so this is a fair, apples-to-apples third data point. class_weight="balanced" matches the tree; depth/leaf constraints are set deliberately, since an unconstrained forest on ~24k rows can still overfit even with bagging.

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np

# Self-contained: redefine prepare_features in case the kernel was restarted
def prepare_features(train_raw, test_raw):
    train_raw, test_raw = train_raw.copy(), test_raw.copy()
    numeric_cols = ["word_count", "char_count", "search_volume", "competition", "cpc", "ctr_prev_30d"]
    categorical_cols = ["content_type", "main_intent"]
    for col in numeric_cols:
        med = train_raw[col].median()
        train_raw[f"{col}_was_missing"] = train_raw[col].isna().astype(int)
        test_raw[f"{col}_was_missing"] = test_raw[col].isna().astype(int)
        train_raw[col] = train_raw[col].fillna(med)
        test_raw[col] = test_raw[col].fillna(med)
    for col in categorical_cols:
        train_raw[col] = train_raw[col].fillna("unknown")
        test_raw[col] = test_raw[col].fillna("unknown")
    feats = (numeric_cols + [f"{c}_was_missing" for c in numeric_cols] + categorical_cols
             + ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
                "content_age_days", "age_tier_order", "days_since_last_update"])
    num_flag_cols = [c for c in feats if c not in categorical_cols]
    tr_dum = pd.get_dummies(train_raw[categorical_cols], prefix=categorical_cols)
    te_dum = pd.get_dummies(test_raw[categorical_cols], prefix=categorical_cols).reindex(columns=tr_dum.columns, fill_value=0)
    Xtr = pd.concat([train_raw[num_flag_cols].reset_index(drop=True), tr_dum.reset_index(drop=True)], axis=1)
    Xte = pd.concat([test_raw[num_flag_cols].reset_index(drop=True), te_dum.reset_index(drop=True)], axis=1)
    return Xtr, train_raw["needs_refresh"].reset_index(drop=True), Xte, test_raw["needs_refresh"].reset_index(drop=True)

gkf = GroupKFold(n_splits=5)
rf_fold_aucs = []
rf_importances_list = []

for fold, (tr_idx, te_idx) in enumerate(gkf.split(df, groups=df["client_id"])):
    Xtr, ytr, Xte, yte = prepare_features(df.iloc[tr_idx], df.iloc[te_idx])

    rf = RandomForestClassifier(
        n_estimators=300, max_depth=6, min_samples_leaf=5,
        class_weight="balanced", random_state=42, n_jobs=-1
    )
    rf.fit(Xtr, ytr)

    fold_auc = roc_auc_score(yte, rf.predict_proba(Xte)[:, 1])
    rf_fold_aucs.append(fold_auc)
    rf_importances_list.append(pd.Series(rf.feature_importances_, index=Xtr.columns))
    print(f"Fold {fold}: RF AUC = {fold_auc:.4f}")

print(f"\nMean RF CV AUC: {np.mean(rf_fold_aucs):.4f}   Std: {np.std(rf_fold_aucs):.4f}")
print(f"Mean tree CV AUC:     {np.mean(fold_aucs):.4f}")
print(f"Mean baseline CV AUC: {np.mean(baseline_fold_aucs):.4f}")

# Artifact check - same discipline as before, every time
avg_importance = pd.concat(rf_importances_list, axis=1).mean(axis=1).sort_values(ascending=False)
print("\nTop 8 avg feature importances (RF, across folds):")
print(avg_importance.head(8))
print(f"\nTop feature share: {avg_importance.iloc[0] / avg_importance.sum():.1%}")

Fold 0: RF AUC = 0.6060
Fold 1: RF AUC = 0.5675
Fold 2: RF AUC = 0.5657
Fold 3: RF AUC = 0.5928
Fold 4: RF AUC = 0.6031

Mean RF CV AUC: 0.5870   Std: 0.0173
Mean tree CV AUC:     0.5752
Mean baseline CV AUC: 0.5699

Top 8 avg feature importances (RF, across folds):
content_age_days          0.203610
age_tier_order            0.106909
impressions_prev_30d      0.087695
clicks_prev_30d           0.073161
days_since_last_update    0.068373
ctr_prev_30d              0.068269
word_count                0.064541
char_count                0.064246
dtype: float64

Top feature share: 20.3%


Stage 11 — Description
Logistic Regression, same folds, same features. Unlike trees, logistic regression is sensitive to feature scale — so StandardScaler is fit on each fold's train data only (never test) before fitting the model, same leakage discipline as everything else.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd
from scipy import stats

logreg_fold_aucs = []

for fold, (tr_idx, te_idx) in enumerate(gkf.split(df, groups=df["client_id"])):
    Xtr, ytr, Xte, yte = prepare_features(df.iloc[tr_idx], df.iloc[te_idx])

    # Logistic regression needs scaled features - fit scaler on TRAIN only
    scaler = StandardScaler()
    Xtr_scaled = scaler.fit_transform(Xtr)
    Xte_scaled = scaler.transform(Xte)  # same train-fit scaler applied to test

    lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
    lr.fit(Xtr_scaled, ytr)

    fold_auc = roc_auc_score(yte, lr.predict_proba(Xte_scaled)[:, 1])
    logreg_fold_aucs.append(fold_auc)
    print(f"Fold {fold}: LogReg AUC = {fold_auc:.4f}")

print(f"\nMean LogReg CV AUC: {np.mean(logreg_fold_aucs):.4f}   Std: {np.std(logreg_fold_aucs):.4f}")

# Final summary across all 4 approaches
print("\n=== Final comparison ===")
print(f"Baseline (v4, rank-based): {np.mean(baseline_fold_aucs):.4f}")
print(f"Decision Tree:             {np.mean(fold_aucs):.4f}")
print(f"Random Forest:             {np.mean(rf_fold_aucs):.4f}")
print(f"Logistic Regression:       {np.mean(logreg_fold_aucs):.4f}")

t, p = stats.ttest_rel(logreg_fold_aucs, baseline_fold_aucs)
print(f"\nLogReg vs baseline: p={p:.3f}")

Fold 0: LogReg AUC = 0.6000
Fold 1: LogReg AUC = 0.5501
Fold 2: LogReg AUC = 0.5559
Fold 3: LogReg AUC = 0.4706
Fold 4: LogReg AUC = 0.5943

Mean LogReg CV AUC: 0.5542   Std: 0.0463

=== Final comparison ===
Baseline (v4, rank-based): 0.5699
Decision Tree:             0.5752
Random Forest:             0.5870
Logistic Regression:       0.5542

LogReg vs baseline: p=0.442


Stage 12 — Description
Error analysis using out-of-fold predictions. Rather than analyze errors on one arbitrary split, generate out-of-fold (OOF) predictions for RF across all 5 folds — every row gets a prediction from a fold where it was in the test set, so this covers the whole dataset with zero leakage, same discipline as before. Then inspect the top-200 ranked pages (a realistic "this week's action list" size): which are true hits, which are false alarms, and what characterizes the pages the model ranks lowest despite actually declining (false negatives) — these tell you where the current feature set is blind.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np

oof_scores = pd.Series(index=df.index, dtype=float)

for fold, (tr_idx, te_idx) in enumerate(gkf.split(df, groups=df["client_id"])):
    Xtr, ytr, Xte, yte = prepare_features(df.iloc[tr_idx], df.iloc[te_idx])
    rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=5,
                                 class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(Xtr, ytr)
    oof_scores.iloc[te_idx] = rf.predict_proba(Xte)[:, 1]

df["rf_oof_score"] = oof_scores

# --- Top 200 ranked pages: true hits vs false alarms ---
top200 = df.sort_values("rf_oof_score", ascending=False).head(200)
true_hits = top200[top200["needs_refresh"] == 1]
false_alarms = top200[top200["needs_refresh"] == 0]
print(f"Top 200: {len(true_hits)} true hits, {len(false_alarms)} false alarms "
      f"(Precision@200 = {len(true_hits)/200:.1%})")

compare_cols = ["content_age_days", "ctr_prev_30d", "clicks_prev_30d", "search_volume", "days_since_last_update"]
print("\nTrue hits vs false alarms - feature medians:")
print(pd.DataFrame({"true_hits": true_hits[compare_cols].median(),
                     "false_alarms": false_alarms[compare_cols].median()}))

# --- Worst false negatives: label=1 but model ranked them lowest ---
false_negs = df[df["needs_refresh"] == 1].sort_values("rf_oof_score").head(200)
true_negs = df[df["needs_refresh"] == 0].sort_values("rf_oof_score").head(200)
print("\nWorst false negatives vs typical true negatives - feature medians:")
print(pd.DataFrame({"false_negatives": false_negs[compare_cols].median(),
                     "true_negatives": true_negs[compare_cols].median()}))

Top 200: 120 true hits, 80 false alarms (Precision@200 = 60.0%)

True hits vs false alarms - feature medians:
                        true_hits  false_alarms
content_age_days            174.0         238.0
ctr_prev_30d                  0.0           0.0
clicks_prev_30d               0.0           0.0
search_volume                 0.0          10.0
days_since_last_update       98.0         103.0

Worst false negatives vs typical true negatives - feature medians:
                        false_negatives  true_negatives
content_age_days                  460.0      445.000000
ctr_prev_30d                        0.0        0.001934
clicks_prev_30d                     0.0        1.000000
search_volume                      10.0       10.000000
days_since_last_update             22.0       22.000000


Stage 12b — Description
Checking the floor-effect columns properly — mean (not just median) and share of exact-zero rows, per group, for the sparse count-type features. This confirms whether "0.0 vs 0.0" truly means no difference, or was hiding something the median couldn't show.

In [ ]:
sparse_cols = ["ctr_prev_30d", "clicks_prev_30d", "search_volume"]

for label, group_a, group_b, name_a, name_b in [
    ("Top200", true_hits, false_alarms, "true_hits", "false_alarms"),
    ("FalseNeg", false_negs, true_negs, "false_negatives", "true_negatives"),
]:
    print(f"\n--- {label} ---")
    for col in sparse_cols:
        print(f"{col:18s} mean {name_a}={group_a[col].mean():.4f}  mean {name_b}={group_b[col].mean():.4f}  "
              f"zero-share {name_a}={(group_a[col]==0).mean():.1%}  zero-share {name_b}={(group_b[col]==0).mean():.1%}")


--- Top200 ---
ctr_prev_30d       mean true_hits=0.0000  mean false_alarms=0.0001  zero-share true_hits=96.7%  zero-share false_alarms=97.5%
clicks_prev_30d    mean true_hits=0.0417  mean false_alarms=0.0500  zero-share true_hits=96.7%  zero-share false_alarms=97.5%
search_volume      mean true_hits=65.8407  mean false_alarms=35.6250  zero-share true_hits=51.7%  zero-share false_alarms=41.2%

--- FalseNeg ---
ctr_prev_30d       mean false_negatives=0.0030  mean true_negatives=0.0039  zero-share false_negatives=56.0%  zero-share true_negatives=47.0%
clicks_prev_30d    mean false_negatives=39.4000  mean true_negatives=18.9850  zero-share false_negatives=56.0%  zero-share true_negatives=47.0%
search_volume      mean false_negatives=60.6417  mean true_negatives=92.0430  zero-share false_negatives=13.5%  zero-share true_negatives=8.0%


Stage 13 — Description
Building the ranked output with reason codes — the actual deliverable format the brief asks for. Priority tiers come from percentile of rf_oof_score (not raw score, since raw scores aren't comparable across models/runs). Reason codes are template strings triggered by the specific patterns found in error analysis — including an explicit, disclosed manual-override flag for the "previously popular, now declining" blind spot, since we know the model under-weights it rather than pretending it doesn't exist.

In [ ]:
import pandas as pd
import numpy as np

output = df.copy()

# --- Priority tier from percentile, not raw score - comparable across runs ---
output["percentile"] = output["rf_oof_score"].rank(pct=True)
output["priority_tier"] = np.select(
    [output["percentile"] >= 0.90, output["percentile"] >= 0.70],
    ["Refresh Immediately", "Monitor"],
    default="No Action"
)

# --- Reason codes, built from what error analysis actually found ---
def build_reasons(row):
    reasons = []
    if row["clicks_prev_30d"] == 0 and row["ctr_prev_30d"] == 0:
        reasons.append("No recent clicks recorded — may be invisible to search, not just declining")
    if row["search_volume"] > df["search_volume"].median() and row["priority_tier"] != "No Action":
        reasons.append("High search opportunity — worth prioritizing over similar lower-volume pages")
    if row["content_age_days"] > df["content_age_days"].quantile(0.75):
        reasons.append("Aging content — oldest quartile of the corpus")
    # Disclosed model-blind-spot override: flag high-prior-traffic pages for manual review
    # even if the model itself scored them lower, since error analysis showed this group
    # is systematically under-flagged
    if row["clicks_prev_30d"] > df["clicks_prev_30d"].median() and row["priority_tier"] == "No Action":
        reasons.append("MANUAL REVIEW FLAG: previously high-traffic page — model may under-detect decline in this group")
    return reasons if reasons else ["No strong signal in either direction"]

output["reason_codes"] = output.apply(build_reasons, axis=1)

# --- Show it in the exact format the original project brief asked for ---
sample = output.sort_values("rf_oof_score", ascending=False).head(3)
for _, row in sample.iterrows():
    print(f"Page: {row['content_id']}")
    print(f"Priority Score: {row['percentile']:.0%}  |  Tier: {row['priority_tier']}")
    print("Reasons:")
    for r in row["reason_codes"]:
        print(f"  - {r}")
    print("-" * 40)

print(f"\nTier distribution:\n{output['priority_tier'].value_counts()}")

Page: content_2432fcb036bb
Priority Score: 100%  |  Tier: Refresh Immediately
Reasons:
  - No recent clicks recorded — may be invisible to search, not just declining
----------------------------------------
Page: content_2a228ce7aa1b
Priority Score: 100%  |  Tier: Refresh Immediately
Reasons:
  - No recent clicks recorded — may be invisible to search, not just declining
----------------------------------------
Page: content_7e1f78c66a44
Priority Score: 100%  |  Tier: Refresh Immediately
Reasons:
  - No recent clicks recorded — may be invisible to search, not just declining
----------------------------------------

Tier distribution:
priority_tier
No Action              18628
Monitor                 5322
Refresh Immediately     2662
Name: count, dtype: int64


Stage 13b — Description
Verifying the top tier isn't one narrow pattern, and correcting reason-code wording if it's describing the wrong thing. Check whether the zero-click top pages have real impressions (confirming "visible but unclicked" rather than "invisible"), and check how many Refresh Immediately pages get more than one reason code — a tier where almost everyone gets exactly one identical reason is a red flag for insufficient differentiation, not a clean result.

In [ ]:
top_tier = output[output["priority_tier"] == "Refresh Immediately"]

# Do these zero-click pages actually have impressions? (visible-but-unclicked vs truly invisible)
zero_click = top_tier[(top_tier["clicks_prev_30d"] == 0) & (top_tier["ctr_prev_30d"] == 0)]
print(f"Zero-click pages in top tier: {len(zero_click)} / {len(top_tier)}")
print(f"Of those, share with impressions_prev_30d > 0: {(zero_click['impressions_prev_30d'] > 0).mean():.1%}")
print(f"Median impressions_prev_30d among zero-click pages: {zero_click['impressions_prev_30d'].median()}")

# How many reasons does each Refresh Immediately page actually get?
reason_counts = top_tier["reason_codes"].apply(len)
print(f"\nReason count distribution in top tier:\n{reason_counts.value_counts().sort_index()}")

# Manual-override flag: how often does it actually fire?
override_count = output["reason_codes"].apply(lambda r: any("MANUAL REVIEW FLAG" in x for x in r)).sum()
print(f"\nManual review flag fired on: {override_count} pages")

Zero-click pages in top tier: 2044 / 2662
Of those, share with impressions_prev_30d > 0: 100.0%
Median impressions_prev_30d among zero-click pages: 65.0

Reason count distribution in top tier:
reason_codes
1    2299
2     361
3       2
Name: count, dtype: int64

Manual review flag fired on: 9757 pages


Stage 13c — Description
Fixing reason-code wording and the override threshold. Wording changes to reflect what we actually confirmed (visible-but-unclicked, not invisible). Override threshold moves from "above median" to a genuine high-traffic bar (90th percentile of clicks_prev_30d), so it only fires on pages that were real previously-popular content — restoring its purpose as a narrow, meaningful flag rather than a blanket one.

In [ ]:
import pandas as pd
import numpy as np

high_traffic_threshold = df["clicks_prev_30d"].quantile(0.90)
print(f"90th percentile clicks_prev_30d (new override bar): {high_traffic_threshold}")

def build_reasons_v2(row):
    reasons = []
    if row["clicks_prev_30d"] == 0 and row["ctr_prev_30d"] == 0 and row["impressions_prev_30d"] > 0:
        reasons.append("Visible in search but not earning clicks — likely a title/metadata/content-match issue, not a visibility problem")
    elif row["clicks_prev_30d"] == 0 and row["impressions_prev_30d"] == 0:
        reasons.append("No recent impressions or clicks — a genuine visibility problem")
    if row["search_volume"] > df["search_volume"].median() and row["priority_tier"] != "No Action":
        reasons.append("High search opportunity — worth prioritizing over similar lower-volume pages")
    if row["content_age_days"] > df["content_age_days"].quantile(0.75):
        reasons.append("Aging content — oldest quartile of the corpus")
    if row["clicks_prev_30d"] >= high_traffic_threshold and row["priority_tier"] == "No Action":
        reasons.append("MANUAL REVIEW FLAG: previously high-traffic page (top 10%) — model may under-detect decline in this group")
    return reasons if reasons else ["No strong signal in either direction"]

output["reason_codes_v2"] = output.apply(build_reasons_v2, axis=1)

override_count_v2 = output["reason_codes_v2"].apply(lambda r: any("MANUAL REVIEW FLAG" in x for x in r)).sum()
print(f"Manual review flag fired on (corrected): {override_count_v2} pages   (was 9757 before the fix)")

# Re-check the same 3 sample pages with the corrected wording
sample = output.sort_values("rf_oof_score", ascending=False).head(3)
for _, row in sample.iterrows():
    print(f"\nPage: {row['content_id']}  |  Reasons:")
    for r in row["reason_codes_v2"]:
        print(f"  - {r}")

90th percentile clicks_prev_30d (new override bar): 12.0
Manual review flag fired on (corrected): 2590 pages   (was 9757 before the fix)

Page: content_2432fcb036bb  |  Reasons:
  - Visible in search but not earning clicks — likely a title/metadata/content-match issue, not a visibility problem

Page: content_2a228ce7aa1b  |  Reasons:
  - Visible in search but not earning clicks — likely a title/metadata/content-match issue, not a visibility problem

Page: content_7e1f78c66a44  |  Reasons:
  - Visible in search but not earning clicks — likely a title/metadata/content-match issue, not a visibility problem


In [ ]:
output[output["clicks_prev_30d"] >= 12]["priority_tier"].value_counts()

,count
priority_tier,
No Action,2590
Monitor,134
Refresh Immediately,13


Stage 14 — Description
Rebuild the page-level frame with client_hash_id included, and check the client panel shape on real data — same diagnostic we ran in Stage 1 on the starter CSV (client count, rows per client, min/max spread), since we already know from dim_clients that some clients have far more history/volume than others, and we shouldn't assume the real warehouse's panel is any more balanced than the starter sample was.

In [ ]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
base = "hf://datasets/FlyRank/internship-warehouse"

print("Connected.")

Connected.


In [ ]:
page_level_full = con.sql(f"""
    WITH first_half AS (
        SELECT client_hash_id, content_hash_id,
               AVG(gsc_impressions) as avg_impressions_h1,
               AVG(gsc_clicks) as avg_clicks_h1,
               SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) as ctr_h1,
               AVG(gsc_avg_position) as avg_position_h1,
               COUNT(*) FILTER (WHERE gsc_impressions > 0) as active_days_h1
        FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE AND report_date <= '2026-03-15'
        GROUP BY client_hash_id, content_hash_id
    ),
    second_half AS (
        SELECT content_hash_id,
               SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) as ctr_h2,
               AVG(gsc_avg_position) as avg_position_h2
        FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE AND report_date > '2026-03-15'
        GROUP BY content_hash_id
    )
    SELECT f.*, s.ctr_h2, s.avg_position_h2
    FROM first_half f
    JOIN second_half s ON f.content_hash_id = s.content_hash_id
""").df()

print(f"Shape: {page_level_full.shape}")

client_counts = page_level_full["client_hash_id"].value_counts()
print(f"\nNumber of unique clients: {page_level_full['client_hash_id'].nunique()}")
print(f"Pages per client - min: {client_counts.min()}, max: {client_counts.max()}, median: {client_counts.median()}")
print(f"\nTop 5 largest clients:\n{client_counts.head()}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Shape: (141467, 9)

Number of unique clients: 43
Pages per client - min: 2, max: 25437, median: 979.0

Top 5 largest clients:
client_hash_id
client_73cda7b4e4f265ea    25437
client_62f4a7e64f5e0096    22894
client_08a6a72ff48e62c0    16164
client_23a62021009f63c4    12801
client_e547b89c05043229     9062
Name: count, dtype: int64


Stage 15 — Description

Build the label on the full warehouse data, then sweep seeds for a balanced client-holdout split — same discipline as Stage 5b, just re-run because the panel shape changed. Label: did the page's average position get worse (h2 > h1)? Split: try several seeds, pick based on train/test label-balance gap alone, before any model exists.

In [ ]:
import numpy as np

# --- Label: same logic as the ML-04 check, now on the full client-tagged frame ---
page_level_full["needs_refresh"] = (page_level_full["avg_position_h2"] > page_level_full["avg_position_h1"]).astype(int)
print(f"Overall label balance: {page_level_full['needs_refresh'].mean():.2%}")

# --- Seed sweep for split balance - picked on label stats, not performance ---
clients = page_level_full["client_hash_id"].unique()
print(f"Total unique clients: {len(clients)}\n")

for seed in [0, 1, 7, 42, 99, 123, 2024]:
    rng = np.random.RandomState(seed)
    test_clients = rng.choice(clients, size=max(1, int(len(clients) * 0.25)), replace=False)

    tr = page_level_full[~page_level_full["client_hash_id"].isin(test_clients)]
    te = page_level_full[page_level_full["client_hash_id"].isin(test_clients)]

    gap = abs(tr["needs_refresh"].mean() - te["needs_refresh"].mean())
    print(f"seed={seed:4d} | train n={len(tr):6d} clients={tr['client_hash_id'].nunique():2d} "
          f"bal={tr['needs_refresh'].mean():.2%} | test n={len(te):6d} clients={te['client_hash_id'].nunique():2d} "
          f"bal={te['needs_refresh'].mean():.2%} | gap={gap:.2%}")

Overall label balance: 54.06%
Total unique clients: 43

seed=   0 | train n=126926 clients=33 bal=53.83% | test n= 14541 clients=10 bal=56.14% | gap=2.32%
seed=   1 | train n=100160 clients=33 bal=51.43% | test n= 41307 clients=10 bal=60.46% | gap=9.04%
seed=   7 | train n=117433 clients=33 bal=55.14% | test n= 24034 clients=10 bal=48.83% | gap=6.30%
seed=  42 | train n= 99390 clients=33 bal=50.96% | test n= 42077 clients=10 bal=61.39% | gap=10.43%
seed=  99 | train n=102806 clients=33 bal=53.07% | test n= 38661 clients=10 bal=56.70% | gap=3.62%
seed= 123 | train n=105088 clients=33 bal=56.31% | test n= 36379 clients=10 bal=47.57% | gap=8.75%
seed=2024 | train n=100382 clients=33 bal=52.41% | test n= 41085 clients=10 bal=58.10% | gap=5.69%


Stage 16 — Description

Lock in seed=0, build the split, and check for missing data in the five features. Same feature set as the ML-04 check (avg_impressions_h1, avg_clicks_h1, ctr_h1, avg_position_h1, active_days_h1) — all confirmed knowable from the first half only. Missing-value handling follows the same train-only-statistics rule as Stage 2, if any turn out to be needed.

In [ ]:
import numpy as np

rng = np.random.RandomState(0)
clients = page_level_full["client_hash_id"].unique()
test_clients = rng.choice(clients, size=max(1, int(len(clients) * 0.25)), replace=False)

train_df2 = page_level_full[~page_level_full["client_hash_id"].isin(test_clients)].copy()
test_df2  = page_level_full[page_level_full["client_hash_id"].isin(test_clients)].copy()

print(f"Train: {train_df2.shape[0]} rows, {train_df2['client_hash_id'].nunique()} clients, bal={train_df2['needs_refresh'].mean():.2%}")
print(f"Test:  {test_df2.shape[0]} rows, {test_df2['client_hash_id'].nunique()} clients, bal={test_df2['needs_refresh'].mean():.2%}")

feature_cols = ["avg_impressions_h1", "avg_clicks_h1", "ctr_h1", "avg_position_h1", "active_days_h1"]

print(f"\nMissing values, train:\n{train_df2[feature_cols].isna().sum()}")
print(f"\nMissing values, test:\n{test_df2[feature_cols].isna().sum()}")

Train: 126926 rows, 33 clients, bal=53.83%
Test:  14541 rows, 10 clients, bal=56.14%

Missing values, train:
avg_impressions_h1    0
avg_clicks_h1         0
ctr_h1                0
avg_position_h1       0
active_days_h1        0
dtype: int64

Missing values, test:
avg_impressions_h1    0
avg_clicks_h1         0
ctr_h1                0
avg_position_h1       0
active_days_h1        0
dtype: int64


Stage 17 — Description

Baseline, built right the first time. Sweep each candidate feature's AUC on train only, keep only real signal (strength > 0.02, same threshold as before), blend with rank-normalization (not min-max, since we know that breaks on skewed count data), then evaluate on test with both Precision@K and AUC, plus the tie-clustering and component-balance checks bundled in from the start rather than found the hard way afterward.

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

candidate_cols = ["avg_impressions_h1", "avg_clicks_h1", "ctr_h1", "avg_position_h1", "active_days_h1"]

# --- Step 1: diagnose signal strength + direction, TRAIN ONLY ---
signal_info = {}
for col in candidate_cols:
    auc = roc_auc_score(train_df2["needs_refresh"], train_df2[col])
    signal_info[col] = {"auc": auc, "flip": auc < 0.5, "strength": abs(auc - 0.5)}

results = pd.DataFrame(signal_info).T.sort_values("strength", ascending=False)
print("Signal diagnosis (train only):")
print(results)

keep_cols = results[results["strength"] > 0.02].index.tolist()
print(f"\nKeeping: {keep_cols}")

# --- Step 2: rank-normalized blend (not min-max - learned that lesson already) ---
def rank_normalize(d, col, source=train_df2):
    sorted_source = np.sort(source[col].values)
    ranks = np.searchsorted(sorted_source, d[col].values, side="right")
    return pd.Series(ranks / len(sorted_source), index=d.index)

def baseline_score(d):
    total = pd.Series(0.0, index=d.index)
    for col in keep_cols:
        r = rank_normalize(d, col)
        total += (1 - r) if signal_info[col]["flip"] else r
    return total / len(keep_cols)

train_df2["baseline_score"] = baseline_score(train_df2)
test_df2["baseline_score"] = baseline_score(test_df2)

# --- Step 3: evaluate + built-in sanity checks ---
def precision_at_k(d, score_col, label_col="needs_refresh", k=50):
    return d.sort_values(score_col, ascending=False).head(k)[label_col].mean()

for k in [25, 50, 100, 200]:
    print(f"Baseline Precision@{k}: {precision_at_k(test_df2, 'baseline_score', k=k):.2%}")

auc = roc_auc_score(test_df2["needs_refresh"], test_df2["baseline_score"])
print(f"\nBaseline ROC-AUC: {auc:.4f}   (test base rate: {test_df2['needs_refresh'].mean():.2%})")

top50 = test_df2.sort_values("baseline_score", ascending=False).head(50)
print(f"Unique scores in top 50: {top50['baseline_score'].nunique()} / 50   (tie-check)")

print("\nComponent balance (tie/dominance check):")
for col in keep_cols:
    r = rank_normalize(test_df2, col)
    comp = (1 - r) if signal_info[col]["flip"] else r
    print(f"{col:22s} std={comp.std():.4f}  corr_w_blend={comp.corr(test_df2['baseline_score']):.3f}")

Signal diagnosis (train only):
                         auc   flip  strength
avg_position_h1     0.353606   True  0.146394
active_days_h1      0.522759  False  0.022759
avg_impressions_h1  0.514844  False  0.014844
ctr_h1              0.510119  False  0.010119
avg_clicks_h1       0.508817  False  0.008817

Keeping: ['avg_position_h1', 'active_days_h1']
Baseline Precision@25: 88.00%
Baseline Precision@50: 94.00%
Baseline Precision@100: 91.00%
Baseline Precision@200: 88.50%

Baseline ROC-AUC: 0.6918   (test base rate: 56.14%)
Unique scores in top 50: 49 / 50   (tie-check)

Component balance (tie/dominance check):
avg_position_h1        std=0.2475  corr_w_blend=0.477
active_days_h1         std=0.3865  corr_w_blend=0.826


Stage 17b — Description

Isolating whether active_days_h1 adds real lift, or is just amplified noise from low cardinality. Compare the single-feature baseline (avg_position_h1 alone) against the 2-feature blend, on identical test data. If the blend doesn't clearly beat the single feature, active_days_h1 isn't earning its place despite technically clearing the 0.02 threshold.

In [ ]:
from sklearn.metrics import roc_auc_score

# Single-feature version: avg_position_h1 only
def single_feature_score(d):
    r = rank_normalize(d, "avg_position_h1")
    return 1 - r  # flip=True for this feature, per Stage 17's diagnosis

train_df2["baseline_score_single"] = single_feature_score(train_df2)
test_df2["baseline_score_single"] = single_feature_score(test_df2)

auc_single = roc_auc_score(test_df2["needs_refresh"], test_df2["baseline_score_single"])
auc_blend = roc_auc_score(test_df2["needs_refresh"], test_df2["baseline_score"])

print(f"Single-feature (avg_position_h1 only) AUC: {auc_single:.4f}")
print(f"Two-feature blend AUC:                     {auc_blend:.4f}")

for k in [25, 50, 100, 200]:
    p_single = precision_at_k(test_df2, "baseline_score_single", k=k)
    p_blend = precision_at_k(test_df2, "baseline_score", k=k)
    print(f"P@{k}: single={p_single:.2%}  blend={p_blend:.2%}")

Single-feature (avg_position_h1 only) AUC: 0.6601
Two-feature blend AUC:                     0.6918
P@25: single=96.00%  blend=88.00%
P@50: single=92.00%  blend=94.00%
P@100: single=96.00%  blend=91.00%
P@200: single=96.50%  blend=88.50%


Stage 18 — Description

All three models, cross-validated on identical GroupKFold folds against the single-feature baseline — one comprehensive comparison this time, since we already know the pattern from the starter data. Models get all 5 raw features (they can properly discount weak ones; the baseline can't). Same per-fold discipline: fit-on-train-only for everything.

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from scipy import stats
import numpy as np
import pandas as pd

feature_cols = ["avg_impressions_h1", "avg_clicks_h1", "ctr_h1", "avg_position_h1", "active_days_h1"]

# Classification metrics below use a fixed 0.5 threshold on predict_proba (and on the
# baseline's [0,1] rank-normalized score) for every model - same cutoff for everyone,
# so the comparison stays fair. This is a reporting choice, not a tuned one: AUC above
# is threshold-free and remains the primary number this study relies on.
THRESH = 0.5

gkf2 = GroupKFold(n_splits=5)
results = {"baseline": [], "tree": [], "rf": [], "logreg": []}
oof_true = {"baseline": [], "tree": [], "rf": [], "logreg": []}
oof_pred = {"baseline": [], "tree": [], "rf": [], "logreg": []}

for fold, (tr_idx, te_idx) in enumerate(gkf2.split(page_level_full, groups=page_level_full["client_hash_id"])):
    tr = page_level_full.iloc[tr_idx]
    te = page_level_full.iloc[te_idx]
    print(f"Fold {fold}: {len(te)} rows, {te['client_hash_id'].nunique()} clients")

    # Baseline: single-feature, rank-normalized on THIS fold's train
    sorted_pos = np.sort(tr["avg_position_h1"].values)
    ranks = np.searchsorted(sorted_pos, te["avg_position_h1"].values, side="right") / len(sorted_pos)
    base_score = 1 - ranks  # flip=True, confirmed in Stage 17
    results["baseline"].append(roc_auc_score(te["needs_refresh"], base_score))
    oof_true["baseline"].append(te["needs_refresh"].values)
    oof_pred["baseline"].append((base_score >= THRESH).astype(int))

    Xtr, ytr = tr[feature_cols], tr["needs_refresh"]
    Xte, yte = te[feature_cols], te["needs_refresh"]

    tree = DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced")
    tree.fit(Xtr, ytr)
    tree_proba = tree.predict_proba(Xte)[:, 1]
    results["tree"].append(roc_auc_score(yte, tree_proba))
    oof_true["tree"].append(yte.values)
    oof_pred["tree"].append((tree_proba >= THRESH).astype(int))

    rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=5,
                                 class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(Xtr, ytr)
    rf_proba = rf.predict_proba(Xte)[:, 1]
    results["rf"].append(roc_auc_score(yte, rf_proba))
    oof_true["rf"].append(yte.values)
    oof_pred["rf"].append((rf_proba >= THRESH).astype(int))

    scaler = StandardScaler()
    Xtr_s, Xte_s = scaler.fit_transform(Xtr), scaler.transform(Xte)
    lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
    lr.fit(Xtr_s, ytr)
    lr_proba = lr.predict_proba(Xte_s)[:, 1]
    results["logreg"].append(roc_auc_score(yte, lr_proba))
    oof_true["logreg"].append(yte.values)
    oof_pred["logreg"].append((lr_proba >= THRESH).astype(int))

print("\n=== Mean CV AUC by model ===")
for name, scores in results.items():
    print(f"{name:10s} mean={np.mean(scores):.4f}  std={np.std(scores):.4f}  folds={[round(s,4) for s in scores]}")

print("\n=== Significance vs baseline ===")
for name in ["tree", "rf", "logreg"]:
    t, p = stats.ttest_rel(results[name], results["baseline"])
    wins = sum(a > b for a, b in zip(results[name], results["baseline"]))
    print(f"{name} vs baseline: mean diff={np.mean(results[name])-np.mean(results['baseline']):.4f}  p={p:.3f}  wins={wins}/5")

# --- Classification metrics at threshold 0.5, pooled out-of-fold (every row scored
# exactly once, by whichever fold held it out) - more stable than averaging per-fold
# metrics on an imbalanced label, and directly comparable across models. ---
print(f"\n=== Classification metrics @ threshold={THRESH}, pooled out-of-fold ===")
for name in results.keys():
    y_true = np.concatenate(oof_true[name])
    y_pred = np.concatenate(oof_pred[name])
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    print(f"\n{name}: accuracy={acc:.4f}  precision={prec:.4f}  recall={rec:.4f}  f1={f1:.4f}")
    print(f"  confusion matrix [[TN FP] [FN TP]]:\n{cm}")


Stage 18b — Description

Checking what tables actually exist in the warehouse, rather than assuming only fact_content_daily_performance and dim_clients are there.

In [ ]:
import duckdb

# List actual folders/tables at the dataset root - don't guess, look
fs = duckdb.sql(f"""
    SELECT DISTINCT regexp_extract(file, 'internship-warehouse/([^/]+)/', 1) as table_name
    FROM glob('{base}/**')
""").df()
print(fs)

                       table_name
0                                
1  fact_content_daily_performance


In [ ]:
from datasets import get_dataset_config_names

configs = get_dataset_config_names("FlyRank/internship-warehouse")
print(configs)

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

['dim_clients', 'dim_content', 'fact_content_daily_performance', 'fact_content_query_90d']


In [ ]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")

# Show only files related to dim_content and fact_content_query_90d
for f in files:
    if "dim_content" in f or "query_90d" in f:
        print(f)

dim_content.parquet
fact_content_query_90d.parquet


In [ ]:
schema_content = con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{base}/dim_content.parquet') LIMIT 1
""").df()
print("=== dim_content ===")
print(schema_content.to_string())

schema_query90d = con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{base}/fact_content_query_90d.parquet') LIMIT 1
""").df()
print("\n=== fact_content_query_90d ===")
print(schema_query90d.to_string())

=== dim_content ===
                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         c

Stage 18c — Description

Verify dim_content's real grain, and check is_deleted/is_published before considering any join — same "prove it, don't assume it" discipline as every table before this one.

Grain check: is content_hash_id alone unique, or do keyword/url add rows per content?

In [ ]:
grain = con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(DISTINCT content_hash_id) as unique_content_ids
    FROM read_parquet('{base}/dim_content.parquet')
""").df()
print("=== Grain check ===")
print(grain)

# is_published / is_deleted breakdown
status = con.sql(f"""
    SELECT is_published, is_deleted, COUNT(*) as n
    FROM read_parquet('{base}/dim_content.parquet')
    GROUP BY is_published, is_deleted
""").df()
print("\n=== Publish/delete status ===")
print(status)

=== Grain check ===
   total_rows  unique_content_ids
0      519606              519606

=== Publish/delete status ===
   is_published  is_deleted       n
0          True       False  411540
1         False       False    6507
2         False        True  101559


Stage 18d — Description

Checking whether dim_content's date fields reflect a single global snapshot that could leak post-decision information. If last_optimized_date/content_updated_date cluster after March 15 in a way that correlates with decline, those columns get excluded as leak risks — same discipline as excluding ctr/avg_position back in Stage 1, just a new mechanism this time (snapshot timing, not window overlap).

Scope to published, non-deleted content only

In [ ]:
dim_content_clean = con.sql(f"""
    SELECT * FROM read_parquet('{base}/dim_content.parquet')
    WHERE is_published IS TRUE AND is_deleted IS FALSE
""").df()
print(f"Content in scope: {len(dim_content_clean)} / 519606")

# Check: how many of these have an optimization date AFTER our decision point (March 15)?
after_cutoff = con.sql("""
    SELECT
        COUNT(*) as total,
        COUNT(*) FILTER (WHERE last_optimized_date > DATE '2026-03-15') as optimized_after_cutoff,
        COUNT(*) FILTER (WHERE content_updated_date > DATE '2026-03-15') as updated_after_cutoff,
        COUNT(*) FILTER (WHERE last_optimized_date IS NULL) as never_optimized
    FROM dim_content_clean
""").df()
print(after_cutoff)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Content in scope: 411540 / 519606
    total  optimized_after_cutoff  updated_after_cutoff  never_optimized
0  411540                   45396                368926           366144


Stage 18e — Description

Join only the columns confirmed safe: structural/immutable content attributes as of March 15, nothing derived from a post-cutoff snapshot.

In [ ]:
safe_content_features = con.sql(f"""
    SELECT
        content_hash_id,
        content_type,
        main_intent,
        word_count,
        char_count,
        search_volume,
        competition,
        cpc,
        provider_used,
        model_used,
        category_count,
        DATE '2026-03-15' - content_created_date as content_age_days_at_cutoff
    FROM read_parquet('{base}/dim_content.parquet')
    WHERE is_published IS TRUE AND is_deleted IS FALSE
      AND content_created_date <= DATE '2026-03-15'
""").df()

print(f"Safe content features: {safe_content_features.shape}")
print(f"Missing values:\n{safe_content_features.isna().sum()}")

# Join onto our existing modeling frame
page_level_enriched = page_level_full.merge(safe_content_features, on="content_hash_id", how="inner")
print(f"\nBefore join: {page_level_full.shape[0]} rows")
print(f"After join:  {page_level_enriched.shape[0]} rows  (inner join drops pages with no in-scope content record)")

Safe content features: (318414, 12)
Missing values:
content_hash_id                    0
content_type                       0
main_intent                    48778
word_count                    106638
char_count                    106638
search_volume                  50212
competition                    50212
cpc                            50212
provider_used                 267783
model_used                     83776
category_count                     0
content_age_days_at_cutoff         0
dtype: int64

Before join: 141467 rows
After join:  141360 rows  (inner join drops pages with no in-scope content record)


Stage 19 — Description

Rebuild the client-holdout split (seed=0, locked) on the enriched frame, handle missing data with the same discipline as Stage 2/16, and re-diagnose signal strength on the expanded feature set — new features may or may not carry real signal, so we check rather than assume any of them help.

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score
import pandas as pd

# Re-split on the enriched frame, same locked seed=0
rng = np.random.RandomState(0)
clients = page_level_enriched["client_hash_id"].unique()
test_clients = rng.choice(clients, size=max(1, int(len(clients) * 0.25)), replace=False)

train_df3 = page_level_enriched[~page_level_enriched["client_hash_id"].isin(test_clients)].copy()
test_df3  = page_level_enriched[page_level_enriched["client_hash_id"].isin(test_clients)].copy()

print(f"Train: {train_df3.shape[0]} rows, bal={train_df3['needs_refresh'].mean():.2%}")
print(f"Test:  {test_df3.shape[0]} rows, bal={test_df3['needs_refresh'].mean():.2%}")

# Missing data - same rules as Stage 2/16: train-only medians + flags, "unknown" for categorical
numeric_new = ["word_count", "char_count", "search_volume", "competition", "cpc", "category_count", "content_age_days_at_cutoff"]
categorical_new = ["content_type", "main_intent", "provider_used", "model_used"]

for col in numeric_new:
    med = train_df3[col].median()
    train_df3[f"{col}_was_missing"] = train_df3[col].isna().astype(int)
    test_df3[f"{col}_was_missing"] = test_df3[col].isna().astype(int)
    train_df3[col] = train_df3[col].fillna(med)
    test_df3[col] = test_df3[col].fillna(med)

for col in categorical_new:
    train_df3[col] = train_df3[col].fillna("unknown")
    test_df3[col] = test_df3[col].fillna("unknown")

print(f"\nRemaining missing, train: {train_df3[numeric_new + categorical_new].isna().sum().sum()}")
print(f"Remaining missing, test:  {test_df3[numeric_new + categorical_new].isna().sum().sum()}")

# Re-diagnose signal strength for the NEW numeric features only (old 5 already characterized in Stage 17)
print("\n=== New feature signal check (train only) ===")
for col in numeric_new:
    auc = roc_auc_score(train_df3["needs_refresh"], train_df3[col])
    print(f"{col:30s} AUC: {auc:.4f}  strength: {abs(auc-0.5):.4f}")

Train: 126821 rows, bal=53.84%
Test:  14539 rows, bal=56.15%

Remaining missing, train: 0
Remaining missing, test:  0

=== New feature signal check (train only) ===
word_count                     AUC: 0.4698  strength: 0.0302
char_count                     AUC: 0.4676  strength: 0.0324
search_volume                  AUC: 0.5212  strength: 0.0212
competition                    AUC: 0.5049  strength: 0.0049
cpc                            AUC: 0.5000  strength: 0.0000
category_count                 AUC: 0.5016  strength: 0.0016
content_age_days_at_cutoff     AUC: 0.5071  strength: 0.0071


Stage 20 — Description

Rebuild the full model comparison on the enriched population, feeding models the original 5 features plus the 3 confirmed new signals (word_count, char_count, search_volume) — deliberately leaving out the four confirmed-noise columns rather than throwing everything at the model, since parsimony was the whole point established back in Stage 8. Baseline stays single-feature (avg_position_h1, rank-normalized) — recomputed fresh on this exact population for a fair fight.

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from scipy import stats
import numpy as np
import pandas as pd

full_feature_cols = ["avg_impressions_h1", "avg_clicks_h1", "ctr_h1", "avg_position_h1", "active_days_h1",
                     "word_count", "char_count", "search_volume"]

# Same fixed 0.5 threshold as Stage 18, for the same reason: comparing models fairly,
# not tuning a cutoff. AUC remains the primary, threshold-free number.
THRESH = 0.5

# Re-run missing-data handling across the WHOLE frame (not just one split) so GroupKFold
# folds each get their own train-only fill, same as Stage 18/prior CV runs
gkf3 = GroupKFold(n_splits=5)
results2 = {"baseline": [], "tree": [], "rf": [], "logreg": []}
oof_true2 = {"baseline": [], "tree": [], "rf": [], "logreg": []}
oof_pred2 = {"baseline": [], "tree": [], "rf": [], "logreg": []}

for fold, (tr_idx, te_idx) in enumerate(gkf3.split(page_level_enriched, groups=page_level_enriched["client_hash_id"])):
    tr = page_level_enriched.iloc[tr_idx].copy()
    te = page_level_enriched.iloc[te_idx].copy()

    for col in ["word_count", "char_count", "search_volume"]:
        med = tr[col].median()
        tr[col] = tr[col].fillna(med)
        te[col] = te[col].fillna(med)

    print(f"Fold {fold}: {len(te)} rows, {te['client_hash_id'].nunique()} clients")

    # Baseline: single-feature rank-normalized position, fit on THIS fold's train
    sorted_pos = np.sort(tr["avg_position_h1"].values)
    ranks = np.searchsorted(sorted_pos, te["avg_position_h1"].values, side="right") / len(sorted_pos)
    base_score = 1 - ranks
    results2["baseline"].append(roc_auc_score(te["needs_refresh"], base_score))
    oof_true2["baseline"].append(te["needs_refresh"].values)
    oof_pred2["baseline"].append((base_score >= THRESH).astype(int))

    Xtr, ytr = tr[full_feature_cols], tr["needs_refresh"]
    Xte, yte = te[full_feature_cols], te["needs_refresh"]

    tree = DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced")
    tree.fit(Xtr, ytr)
    tree_proba = tree.predict_proba(Xte)[:, 1]
    results2["tree"].append(roc_auc_score(yte, tree_proba))
    oof_true2["tree"].append(yte.values)
    oof_pred2["tree"].append((tree_proba >= THRESH).astype(int))

    rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=5,
                                 class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(Xtr, ytr)
    rf_proba = rf.predict_proba(Xte)[:, 1]
    results2["rf"].append(roc_auc_score(yte, rf_proba))
    oof_true2["rf"].append(yte.values)
    oof_pred2["rf"].append((rf_proba >= THRESH).astype(int))

    scaler = StandardScaler()
    Xtr_s, Xte_s = scaler.fit_transform(Xtr), scaler.transform(Xte)
    lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
    lr.fit(Xtr_s, ytr)
    lr_proba = lr.predict_proba(Xte_s)[:, 1]
    results2["logreg"].append(roc_auc_score(yte, lr_proba))
    oof_true2["logreg"].append(yte.values)
    oof_pred2["logreg"].append((lr_proba >= THRESH).astype(int))

print("\n=== Mean CV AUC (with content features added) ===")
for name, scores in results2.items():
    print(f"{name:10s} mean={np.mean(scores):.4f}  std={np.std(scores):.4f}")

print("\n=== Significance vs baseline ===")
for name in ["tree", "rf", "logreg"]:
    t, p = stats.ttest_rel(results2[name], results2["baseline"])
    wins = sum(a > b for a, b in zip(results2[name], results2["baseline"]))
    print(f"{name} vs baseline: mean diff={np.mean(results2[name])-np.mean(results2['baseline']):.4f}  p={p:.3f}  wins={wins}/5")

print("\n=== Compare to Stage 18 (features WITHOUT content data) ===")
print(f"RF before content features: 0.6379  |  RF with content features: {np.mean(results2['rf']):.4f}")

# --- Classification metrics at threshold 0.5, pooled out-of-fold - same method as
# Stage 18, so the two feature sets are directly comparable line for line. ---
print(f"\n=== Classification metrics @ threshold={THRESH}, pooled out-of-fold ===")
for name in results2.keys():
    y_true = np.concatenate(oof_true2[name])
    y_pred = np.concatenate(oof_pred2[name])
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    print(f"\n{name}: accuracy={acc:.4f}  precision={prec:.4f}  recall={rec:.4f}  f1={f1:.4f}")
    print(f"  confusion matrix [[TN FP] [FN TP]]:\n{cm}")


Stage 21 — Description

Error analysis on the chosen baseline (single-feature, position-based), using out-of-fold scores across all 5 GroupKFold folds so every page gets evaluated fairly, then straight into the ranked output with reason codes.

In [ ]:
import numpy as np
import pandas as pd

# --- Out-of-fold baseline scores across the whole population ---
oof_baseline = pd.Series(index=page_level_full.index, dtype=float)

for fold, (tr_idx, te_idx) in enumerate(gkf2.split(page_level_full, groups=page_level_full["client_hash_id"])):
    tr = page_level_full.iloc[tr_idx]
    te_idx_actual = page_level_full.index[te_idx]
    sorted_pos = np.sort(tr["avg_position_h1"].values)
    ranks = np.searchsorted(sorted_pos, page_level_full.loc[te_idx_actual, "avg_position_h1"].values, side="right") / len(sorted_pos)
    oof_baseline.loc[te_idx_actual] = 1 - ranks  # flip=True, confirmed Stage 17

page_level_full["oof_score"] = oof_baseline

# --- Error analysis: top 200 (a realistic weekly action list) ---
top200 = page_level_full.sort_values("oof_score", ascending=False).head(200)
true_hits = top200[top200["needs_refresh"] == 1]
false_alarms = top200[top200["needs_refresh"] == 0]
print(f"Top 200: {len(true_hits)} true hits, {len(false_alarms)} false alarms (P@200 = {len(true_hits)/200:.1%})")

compare_cols = ["avg_impressions_h1", "avg_clicks_h1", "ctr_h1", "active_days_h1"]
print("\nTrue hits vs false alarms - medians:")
print(pd.DataFrame({"true_hits": true_hits[compare_cols].median(), "false_alarms": false_alarms[compare_cols].median()}))

# --- Worst false negatives (real decliners ranked lowest) vs true negatives ---
false_negs = page_level_full[page_level_full["needs_refresh"] == 1].sort_values("oof_score").head(200)
true_negs = page_level_full[page_level_full["needs_refresh"] == 0].sort_values("oof_score").head(200)
print("\nFalse negatives vs true negatives - medians:")
print(pd.DataFrame({"false_negatives": false_negs[compare_cols].median(), "true_negatives": true_negs[compare_cols].median()}))

# --- Ranked output with reason codes ---
output2 = page_level_full.copy()
output2["percentile"] = output2["oof_score"].rank(pct=True)
output2["priority_tier"] = np.select(
    [output2["percentile"] >= 0.90, output2["percentile"] >= 0.70],
    ["Refresh Immediately", "Monitor"], default="No Action"
)

def build_reasons(row):
    reasons = []
    if row["avg_clicks_h1"] == 0 and row["avg_impressions_h1"] > 0:
        reasons.append("Visible in search but earning zero clicks in the prior window")
    if row["active_days_h1"] < 8 and row["priority_tier"] != "No Action":
        reasons.append("Inconsistent visibility - fewer than half of tracked days had impressions")
    if row["avg_position_h1"] > page_level_full["avg_position_h1"].median():
        reasons.append("Already ranking below the corpus median position")
    return reasons if reasons else ["Ranking position is the primary driver - no secondary flags"]

output2["reason_codes"] = output2.apply(build_reasons, axis=1)
print(f"\nTier distribution:\n{output2['priority_tier'].value_counts()}")

sample = output2.sort_values("oof_score", ascending=False).head(3)
for _, row in sample.iterrows():
    print(f"\nPage: {row['content_hash_id']} | Tier: {row['priority_tier']} | Reasons: {row['reason_codes']}")

Top 200: 188 true hits, 12 false alarms (P@200 = 94.0%)

True hits vs false alarms - medians:
                    true_hits  false_alarms
avg_impressions_h1        1.0           1.0
avg_clicks_h1             0.0           0.0
ctr_h1                    0.0           0.0
active_days_h1            1.0           1.0

False negatives vs true negatives - medians:
                    false_negatives  true_negatives
avg_impressions_h1         2.354167             1.0
avg_clicks_h1              0.000000             0.0
ctr_h1                     0.000000             0.0
active_days_h1             9.500000             2.0

Tier distribution:
priority_tier
No Action              99027
Monitor                28293
Refresh Immediately    14147
Name: count, dtype: int64

Page: content_25c92ae6b3af84e8 | Tier: Refresh Immediately | Reasons: ['Visible in search but earning zero clicks in the prior window', 'Inconsistent visibility - fewer than half of tracked days had impressions']

Page: content_340a